# RSNA Knee: Fluid Specialist 5-Fold Training Pipeline
Pretrains on all 4,407 studies with soft labels, then trains 5-fold cross-validated checkpoints targeting Synovitis (3.5x), Effusion (2.0x), and Baker's Cyst (2.5x).

## Datasets
- **Metadata & Pseudo-Labels CSV:** [RSNA Knee Metadata & Pseudo-Labels](https://www.kaggle.com/datasets/barun2104/rsna-knee-stratified-folds-and-llm-soft-labels)
- **Preprocessed 3D Volumes:** [RSNA Knee Processed 3D Volumes](https://www.kaggle.com/datasets/barun2104/rsna-knee-mri-processed-3d-volumes)

## 1. Environment Setup & Data Verification
Let's verify GPU availability and confirm that our preprocessed `(24, 224, 224)` `.npz` volume files are properly mounted in `/kaggle/input/`.

In [ ]:
import os
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.metrics import roc_auc_score

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Using compute device: {device}")

# Path Specifications
IMG_DIR = "/kaggle/input/datasets/barun2104/rsna-knee-mri-processed-3d-volumes"
CSV_PATH = "/kaggle/input/datasets/barun2104/rsna-knee-stratified-folds-and-llm-soft-labels/train_folds_with_pseudo.csv"
TRAIN_SERIES_PATH = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"

# Verify File Count in Input Directory
available_files = len([f for f in os.listdir(IMG_DIR) if f.endswith('.npz')])
print(f"✅ Input directory mounted: {available_files:,} volume files (.npz) detected.")

## 2. Load Metadata & Generate Study-Series Mapping 
Now, let's load our cross-validation metadata file containing pre-assigned stratified folds and target annotations across **12 primary knee pathologies.** We will use the `train_series.csv` to generate mapping between Study and Series.

In [ ]:
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

df = pd.read_csv(CSV_PATH)
train_series_df = pd.read_csv(TRAIN_SERIES_PATH)

study_to_series_map = train_series_df.groupby('StudyInstanceUID')['SeriesInstanceUID'].apply(list).to_dict()
df['series_ids'] = df['StudyInstanceUID'].map(study_to_series_map)
df = df[df['series_ids'].notna() & (df['series_ids'].map(len) > 0)].reset_index(drop=True)

# Impute soft labels for all unannotated studies
for col in TARGET_COLS:
    candidates = [f"pseudo_{col}", f"pseudo_{col.replace(' ', '_')}", f"pseudo_{col.replace(' ', '_').lower()}", f"{col}_pseudo"]
    for cand in candidates:
        if cand in df.columns:
            df[col] = df[col].fillna(df[cand])
            break
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mean())

expert_mask = (df['fold'] != -1)
print(f"Total Studies: {len(df):,} (Expert Ground Truth: {expert_mask.sum()}, Weak/Pseudo: {(~expert_mask).sum()})")
print("Expert Fold distribution:")
print(df[expert_mask]['fold'].value_counts().sort_index())


## 3. PyTorch Dataset & DataLoader
Our dataset loader fetches each preprocessed `(24, 224, 224)` volume, normalizes pixel values from `uint8` (0-255) to `float32` (0.0-1.0), and returns the 12-element multi-label target vector.

In [ ]:
class KneeMRIDataset(Dataset):
    def __init__(self, df, series_map, img_dir=IMG_DIR, target_slices=24, is_train=True):
        self.df = df.reset_index(drop=True)
        self.series_map = series_map
        self.img_dir = img_dir
        self.target_slices = target_slices
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        series_ids = self.series_map.get(study_id, [])
        loaded_vols = []
        for s_id in series_ids:
            series_path = os.path.join(self.img_dir, f"{s_id}.npz")
            if os.path.exists(series_path):
                try:
                    with np.load(series_path) as npz:
                        key = 'data' if 'data' in npz else npz.files[0]
                        vol = npz[key]
                        loaded_vols.append(vol)
                except Exception:
                    pass

        if len(loaded_vols) == 0:
            combined_vol = np.zeros((self.target_slices, 224, 224), dtype=np.float32)
        else:
            combined_vol = np.concatenate(loaded_vols, axis=0)
            curr_slices = combined_vol.shape[0]
            if curr_slices != self.target_slices:
                slice_indices = np.linspace(0, curr_slices - 1, self.target_slices).astype(int)
                combined_vol = combined_vol[slice_indices]

        tensor_vol = torch.from_numpy(combined_vol).float()
        if tensor_vol.max() > 1.0:
            tensor_vol = tensor_vol / 255.0

        if self.is_train and np.random.rand() > 0.5:
            tensor_vol = torch.flip(tensor_vol, dims=[-1])

        labels = torch.tensor(row[TARGET_COLS].values.astype(np.float32))
        return tensor_vol, labels


## 4. 2.5D ResNet-34 Architecture
Instead of computationally expensive 3D convolutions, we adopt a **2.5D approach:**
- We treat the **24 depth slices** as input feature channels into a standard 2D Convolutional Backbone.
- The first layer (`conv1`) is adapted from 3 channels to 24 channels by repeating ResNet pre-trained weights across the depth axis, maintaining rich transfer-learning initialization.

In [ ]:
class KneeResNet25D(nn.Module):
    def __init__(self, num_classes=12, in_channels=24, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet34(weights=weights)
        
        # Modify conv1 for 24 depth slices
        old_conv = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(
            in_channels, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias
        )
        
        # Proper weight expansion initialization
        with torch.no_grad():
            repeat_factor = (in_channels // 3) + 1
            new_weight = old_conv.weight.repeat(1, repeat_factor, 1, 1)[:, :in_channels, :, :]
            # Scale by sqrt ratio to maintain variance
            self.backbone.conv1.weight = nn.Parameter(new_weight * np.sqrt(3.0 / in_channels))
            
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

model = KneeResNet25D(num_classes=len(TARGET_COLS), in_channels=24, pretrained=True).to(device)
print(f"✅ Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters.")

## 5. Metric Evaluation & Loss Setup
We evaluate validation performance using **Macro ROC-AUC** across all 12 targets, handling edge cases where a batch or split might have only single-class instances.

In [ ]:
import torch.nn.functional as F

class WeightedFocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weights, gamma=1.5):
        super().__init__()
        self.register_buffer('pos_weights', torch.tensor(pos_weights, dtype=torch.float32))
        self.gamma = gamma
        
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        loss = focal_weight * self.pos_weights * bce
        return loss.mean()

def compute_macro_auc(y_true, y_pred):
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) > 1:
            auc = roc_auc_score(y_true[:, i], y_pred[:, i])
            aucs.append(auc)
    return float(np.mean(aucs)) if len(aucs) > 0 else 0.5

# Focus weights: Synovitis (3.5x), Effusion (2.0x), Baker's (2.5x)
CLASS_WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 3.5, 2.5, 1.0, 1.0]
criterion = WeightedFocalBCEWithLogitsLoss(CLASS_WEIGHTS, gamma=1.5).to(device)


## 6. Training & Validation Loop

In [ ]:
print("=== Starting 5-Fold Fluid Specialist Training ===")

# Step 1: Rapid representation pretraining on all 4,407 studies
print(">> Step 1: Pre-training on full dataset (4,407 studies) for 3 epochs...")
pretrain_dataset = KneeMRIDataset(df, series_map=study_to_series_map, is_train=True)
pretrain_loader = DataLoader(pretrain_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

opt_pre = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scaler = torch.amp.GradScaler('cuda')

for ep in range(1, 4):
    model.train()
    running_loss = 0.0
    for x_batch, y_batch in tqdm(pretrain_loader, desc=f"Pretrain Epoch {ep}/3"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        opt_pre.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(x_batch), y_batch)
        scaler.scale(loss).backward()
        scaler.step(opt_pre)
        scaler.update()
        running_loss += loss.item() * x_batch.size(0)
    print(f"Pretrain Epoch {ep:02d} | Loss: {running_loss / len(pretrain_dataset):.4f}")

torch.save(model.state_dict(), "/kaggle/working/fluid_specialist_pretrain.pt")
print("Saved pre-trained backbone checkpoint.")

# Step 2: 5-Fold Cross Validation Fine-Tuning
print("\n>> Step 2: Fine-Tuning on 5 Folds...")
manual_df = df[df['fold'] != -1].reset_index(drop=True)
fold_scores = []

for fold_idx in range(5):
    print(f"\n--- Fold {fold_idx}/4 Training ---")
    # Fresh clone of pretrained model for each fold
    fold_model = KneeResNet25D(num_classes=len(TARGET_COLS), in_channels=24, pretrained=False).to(device)
    fold_model.load_state_dict(torch.load("/kaggle/working/fluid_specialist_pretrain.pt"))
    
    tr_df = manual_df[manual_df['fold'] != fold_idx].reset_index(drop=True)
    va_df = manual_df[manual_df['fold'] == fold_idx].reset_index(drop=True)
    
    tr_loader = DataLoader(KneeMRIDataset(tr_df, study_to_series_map, is_train=True), batch_size=16, shuffle=True, num_workers=2)
    va_loader = DataLoader(KneeMRIDataset(va_df, study_to_series_map, is_train=False), batch_size=16, shuffle=False, num_workers=2)
    
    opt = torch.optim.AdamW(fold_model.parameters(), lr=1e-4, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5, eta_min=1e-6)
    
    best_f_auc = 0.0
    best_ckpt_path = f"/kaggle/working/fluid_specialist_f{fold_idx}.pt"
    
    for ep in range(1, 6):
        fold_model.train()
        for x_b, y_b in tr_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                l = criterion(fold_model(x_b), y_b)
            scaler.scale(l).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()
        
        fold_model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for x_b, y_b in va_loader:
                x_b = x_b.to(device)
                with torch.amp.autocast('cuda'):
                    out = torch.sigmoid(fold_model(x_b))
                preds.append(out.cpu().numpy())
                targets.append(y_b.numpy())
        val_auc = compute_macro_auc(np.vstack(targets), np.vstack(preds))
        if val_auc > best_f_auc:
            best_f_auc = val_auc
            torch.save(fold_model.state_dict(), best_ckpt_path)
            
    print(f"Fold {fold_idx} Complete | Best Val Macro AUC: {best_f_auc:.4f} -> Saved {best_ckpt_path}")
    fold_scores.append(best_f_auc)

print(f"\n=== 5-Fold Training Complete! Mean Val Macro AUC: {np.mean(fold_scores):.4f} ===")


## 7. Convergence Plotting

In [ ]:
import glob
print('Generated Model Checkpoints:')
for p in sorted(glob.glob('/kaggle/working/*.pt')):
    print(f' - {p} ({os.path.getsize(p)/1e6:.1f} MB)')


## 8. Next Improvements & Roadmap
- **Semi-Supervised Pre-Training:** Pre-train on the unannotated studies (`fold == -1`) using the soft probability targets (`pseudo_*`) provided in the metadata CSV before fine-tuning on ground-truth folds.
- **Backbone Experimentation:** Test heavier backbones like EfficientNet-B4 or ConvNeXt.
- **3D Data Augmentation:** Introduce spatial transforms such as random horizontal flipping, slight rotations, and intensity gamma shifts via `albumentations`.